In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchvision.models.resnet import ResNet, BasicBlock
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [1]:
import torch
from torchvision import datasets, transforms

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),         # resize images to 32x32
    transforms.ToTensor(),               # convert image to tensor
    transforms.Normalize((0.5,), (0.5,)) # normalize pixel values
])

# Load CIFAR-10 dataset
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)


100.0%


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [7]:
labels = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck"
}

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import numpy as np

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10
train_data = datasets.CIFAR10(root="./data", train=True, download=True)

# Filter for cat and dog
cat_label = 3
dog_label = 5
targets_array = np.array(train_data.targets)
cat_dog_indices = np.where((targets_array == cat_label) | (targets_array == dog_label))[0]
filtered_images = train_data.data[cat_dog_indices]
filtered_labels = np.array([0 if label==cat_label else 1 for label in targets_array[cat_dog_indices]])

# Custom Dataset
class CatDogDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Dataset & DataLoader
cat_dog_train = CatDogDataset(filtered_images, filtered_labels, transform=transform)
train_loader = DataLoader(cat_dog_train, batch_size=32, shuffle=True)

# Test batch
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels[:10]}")



Files already downloaded and verified
Batch shape: torch.Size([32, 3, 32, 32]), Labels: tensor([1, 0, 0, 0, 1, 1, 0, 0, 1, 0])


In [4]:
from torch.utils.data import Dataset

class CatDogDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert to PIL Image before applying transform
        from PIL import Image
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        
        return image, label


In [5]:
# Create CatDogDataset with both images and labels
cat_dog_train = CatDogDataset(filtered_images, filtered_labels, transform=transform)

# Print total number of images
print("Total images:", len(cat_dog_train))


Total images: 10000


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(cat_dog_train, batch_size=32, shuffle=True)


In [9]:
# --- Create the Test Set and Loader ---
print("Creating the test set...")

# Get the raw test data (this variable was loaded by Group 1)
test_targets_np = np.array(test_data.targets)

# Find the indices for cats (3) and dogs (5)
test_cat_dog_indices = np.where((test_targets_np == 3) | (test_targets_np == 5))[0]

# Filter the data and labels
test_filtered_images = test_data.data[test_cat_dog_indices]
test_filtered_labels_raw = test_targets_np[test_cat_dog_indices]

# Remap labels: cat (3) -> 1, dog (5) -> 0
test_filtered_labels = np.where(test_filtered_labels_raw == 3, 1, 0)

# Create the test dataset instance
cat_dog_test = CatDogDataset(test_filtered_images, test_filtered_labels, transform=transform)

print(f"Test set created with {len(cat_dog_test)} images.")

# Create the Test Loader
BATCH_SIZE = 64 # Using the same batch size as the train_loader
test_loader = DataLoader(cat_dog_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Test loader created with batch size {BATCH_SIZE}.")
print("All data is now loaded. Ready to build the model.")

Creating the test set...
Test set created with 2000 images.
Test loader created with batch size 64.
All data is now loaded. Ready to build the model.


In [2]:
# --- Load, Modify, and Freeze ResNet18 ---

print("\nLoading pretrained ResNet18 model...")
# We use the recommended 'DEFAULT' weights (ImageNetV1)
model = resnet18(weights=ResNet18_Weights.DEFAULT)


# --- Modify the final FC layer for binary classification ---
# Get the number of input features for the original 'fc' layer
num_ftrs = model.fc.in_features
# Replace the final layer with a new one that has only 1 output neuron
model.fc = nn.Linear(num_ftrs, 1)
print(f"ResNet18 final 'fc' layer modified for 1 output.")


# --- Freeze all layers EXCEPT layer3, layer4, and fc ---
# First, freeze all parameters in the model
for param in model.parameters():
    param.requires_grad = False

# Now, unfreeze *only* the layers you need to fine-tune
print("Unfreezing layers for fine-tuning: 'layer3', 'layer4', and 'fc'")
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True
    

# --- Verification Step ---
print("\n--- Trainable Parameters Verification ---")
total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"  [TRAINABLE] {name}")

print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nModel setup is complete.")
print("The real data is loaded and the model is ready for fine-tuning.")


Loading pretrained ResNet18 model...
ResNet18 final 'fc' layer modified for 1 output.
Unfreezing layers for fine-tuning: 'layer3', 'layer4', and 'fc'

--- Trainable Parameters Verification ---
  [TRAINABLE] layer3.0.conv1.weight
  [TRAINABLE] layer3.0.bn1.weight
  [TRAINABLE] layer3.0.bn1.bias
  [TRAINABLE] layer3.0.conv2.weight
  [TRAINABLE] layer3.0.bn2.weight
  [TRAINABLE] layer3.0.bn2.bias
  [TRAINABLE] layer3.0.downsample.0.weight
  [TRAINABLE] layer3.0.downsample.1.weight
  [TRAINABLE] layer3.0.downsample.1.bias
  [TRAINABLE] layer3.1.conv1.weight
  [TRAINABLE] layer3.1.bn1.weight
  [TRAINABLE] layer3.1.bn1.bias
  [TRAINABLE] layer3.1.conv2.weight
  [TRAINABLE] layer3.1.bn2.weight
  [TRAINABLE] layer3.1.bn2.bias
  [TRAINABLE] layer4.0.conv1.weight
  [TRAINABLE] layer4.0.bn1.weight
  [TRAINABLE] layer4.0.bn1.bias
  [TRAINABLE] layer4.0.conv2.weight
  [TRAINABLE] layer4.0.bn2.weight
  [TRAINABLE] layer4.0.bn2.bias
  [TRAINABLE] layer4.0.downsample.0.weight
  [TRAINABLE] layer4.0.d